### AsyncIO
#### Основные понятия

In [ ]:
import asyncio

# Корутина — определяется через async def
async def greet(name: str, delay: float) -> str:
    await asyncio.sleep(delay)   # неблокирующее ожидание (освобождает цикл событий)
    return f"Привет, {name}!"

# Запуск одной корутины
result = asyncio.run(greet("Alice", 1.0))
print(result)  # Привет, Alice!

#### `asyncio.gather` — конкурентное выполнение

In [ ]:
import asyncio
import time

async def fetch_country_gdp(country: str) -> dict:
    """Симуляция API-запроса к макроэкономической базе данных."""
    delays = {"DE": 0.5, "FR": 0.8, "IT": 1.2}
    await asyncio.sleep(delays.get(country, 1.0))
    gdp_data = {"DE": 4.07, "FR": 2.78, "IT": 2.10}
    return {"country": country, "gdp_tn_usd": gdp_data[country]}

async def main():
    countries = ["DE", "FR", "IT"]
    start = time.perf_counter()

    # Последовательно — общее время = 0.5 + 0.8 + 1.2 = 2.5с
    # results = [await fetch_country_gdp(c) for c in countries]

    # Конкурентно с gather — общее время ≈ max(0.5, 0.8, 1.2) = 1.2с
    results = await asyncio.gather(*[fetch_country_gdp(c) for c in countries])

    elapsed = time.perf_counter() - start
    print(f"Получено данных по {len(results)} странам за {elapsed:.2f}с")
    print(results)

asyncio.run(main())

In [ ]:
import asyncio
import aiohttp  # pip install aiohttp

async def fetch_url(session: aiohttp.ClientSession, url: str) -> str:
    async with session.get(url) as response:
        return await response.text()

async def fetch_all(urls: list[str]) -> list[str]:
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_url(session, url) for url in urls]
        return await asyncio.gather(*tasks)

# urls = ["https://api.worldbank.org/v2/country/DE/indicator/NY.GDP.MKTP.CD?format=json", ...]
# results = asyncio.run(fetch_all(urls))

#### Обработка ошибок в gather

In [ ]:
async def risky_fetch(i: int):
    await asyncio.sleep(0.1)
    if i == 2:
        raise ValueError(f"Задача {i} завершилась с ошибкой")
    return i * 10

async def main():
    # return_exceptions=True — сбои не отменяют другие задачи
    results = await asyncio.gather(
        *[risky_fetch(i) for i in range(5)],
        return_exceptions=True
    )
    for i, r in enumerate(results):
        if isinstance(r, Exception):
            print(f"Ошибка задачи {i}: {r}")
        else:
            print(f"Результат задачи {i}: {r}")

asyncio.run(main())

### Многопроцессорность
#### Базовый `Process`

In [ ]:
from multiprocessing import Process

def worker(n: int) -> None:
    result = sum(i**2 for i in range(n))
    print(f"Сумма квадратов до {n}: {result}")

if __name__ == "__main__":  # ОБЯЗАТЕЛЬНО — требуется в Windows/macOS
    p = Process(target=worker, args=(1_000_000,))
    p.start()
    p.join()   # ждать завершения

#### `Pool.map` — параллелизация по списку

In [ ]:
from multiprocessing import Pool
import time

def compute_stats(filepath: str) -> dict:
    """CPU-ограниченная: читаем файл + вычисляем статистику."""
    import csv
    with open(filepath) as f:
        amounts = [float(row["amount"]) for row in csv.DictReader(f)]
    return {
        "file": filepath,
        "total": sum(amounts),
        "count": len(amounts),
        "avg": sum(amounts) / len(amounts),
    }

if __name__ == "__main__":
    files = [f"data/batch_{i:03d}.csv" for i in range(20)]

    # Последовательно
    t0 = time.perf_counter()
    sequential = [compute_stats(f) for f in files]
    print(f"Последовательно: {time.perf_counter() - t0:.2f}с")

    # Параллельно (использует все ядра CPU)
    t0 = time.perf_counter()
    with Pool() as pool:
        parallel = pool.map(compute_stats, files)
    print(f"Параллельно ({Pool()._processes} ядер): {time.perf_counter() - t0:.2f}с")

#### `imap` — ленивая итерация для больших входных данных

In [ ]:
from multiprocessing import Pool

def process_row(row: dict) -> float:
    return float(row["amount"]) * 1.2  # некоторое преобразование

if __name__ == "__main__":
    with Pool(processes=4) as pool:
        # imap ленивый — не загружает все результаты в память сразу
        for result in pool.imap(process_row, large_row_list, chunksize=500):
            # обрабатываем каждый результат по мере поступления
            pass

#### Накладные расходы на сериализацию — когда многопроцессорность не оправдана

In [ ]:
# Передача данных между процессами требует сериализации через pickle
# Маленькие данные: накладные расходы > ускорение
# Практическое правило: если каждая задача < 100 мс, рассмотрите векторизацию

import pickle
import time

big_array = list(range(1_000_000))

t0 = time.perf_counter()
serialized = pickle.dumps(big_array)
deserialized = pickle.loads(serialized)
print(f"Pickle туда-обратно: {time.perf_counter() - t0:.3f}с")
# Если ваша задача занимает меньше этого времени, многопроцессорность не поможет

**Задание:**
1. Написать `async`-функцию, которая получает экономические показатели для 20 стран из mock-API (предоставляется как локальный сервер).
2. Использовать `asyncio.gather` с `return_exceptions=True` и сообщить, по каким странам возникли ошибки.
3. Для шага парсинга и агрегации использовать `multiprocessing.Pool.map` и сравнить время с последовательным выполнением.

#### Три версии одной задачи

In [ ]:
# Задача: «Скачать 50 файлов и посчитать строки в каждом»
# Симулируется с помощью sleep(0.1) для скачивания и вычислений

import asyncio
import time
from multiprocessing import Pool

# --- Версия 1: Синхронная ---
def download_sync(i):
    time.sleep(0.1)   # симуляция I/O
    return sum(range(10_000))  # симуляция CPU

def run_sync(n=10):
    t0 = time.perf_counter()
    results = [download_sync(i) for i in range(n)]
    print(f"Синхронно: {time.perf_counter() - t0:.2f}с")

# --- Версия 2: AsyncIO (побеждает при I/O-ограниченных задачах) ---
async def download_async(i):
    await asyncio.sleep(0.1)   # симуляция неблокирующего I/O
    return i * 100

async def run_async(n=10):
    t0 = time.perf_counter()
    results = await asyncio.gather(*[download_async(i) for i in range(n)])
    print(f"Асинхронно: {time.perf_counter() - t0:.2f}с")

# --- Версия 3: Многопроцессорность (побеждает при CPU-ограниченных задачах) ---
def cpu_task(i):
    return sum(range(1_000_000))  # чистый CPU

def run_multiprocessing(n=10):
    t0 = time.perf_counter()
    with Pool() as pool:
        results = pool.map(cpu_task, range(n))
    print(f"Многопроцессорность: {time.perf_counter() - t0:.2f}с")

if __name__ == "__main__":
    run_sync(10)
    asyncio.run(run_async(10))
    run_multiprocessing(10)